# Lab Day 2 — Backbone, Công Thức Huấn Luyện và Suy Luận Trên DeepWeeds

- **Học viên:** Dương Văn Thành (MSSV: 2A202602368)
- **Lớp / Track:** Track 4 - Advanced Deep Learning (VinAI & Đại học Công nghệ)
- **Tập dữ liệu:** DeepWeeds (Olsen et al., 2019) — 17.509 ảnh 9 lớp (Fold 0)

Bản đồ thực nghiệm: Bước 0 (EDA & Pipeline sanity check) → Bước 1 (So sánh >= 5 Backbone) → Bước 2 (Ablation công thức >= 3 trục) → Bước 3 (Suy luận & Benchmark độ trễ) → Bước 4 (Chung kết 3 seeds + `eval.py grade`) → Bước 5 (Tổng kết sản phẩm).

## 0. Cài đặt môi trường & Kiểm tra GPU

In [ ]:
import sys, os, platform
from pathlib import Path
import numpy as np, pandas as pd, torch, timm

# Thiết lập thư mục làm việc và đường dẫn code
CODE_DIR = Path(".").resolve()
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
ROOT_DIR = CODE_DIR.parent.parent.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print("Python:", platform.python_version(), "| PyTorch:", torch.__version__, "| timm:", timm.__version__)
print("Thiết bị:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

IMAGES_DIR = "data/images"
LABELS_DIR = "data/labels"

## Bước 0 — Khảo Sát Dữ Liệu (EDA) & Kiểm Chuẩn Pipeline (Sanity Checks)

In [ ]:
import dataset as ds
import eval as ev

# 1. Đọc và kiểm định phân chia Fold 0 chuẩn xác
train_df, val_df, test_df = ds.load_split(LABELS_DIR, fold=0)
stats = ds.check_split(train_df, val_df, test_df, Path(IMAGES_DIR))

# 2. In thống kê phân bố từng lớp đối chiếu Table 1 Olsen et al.
df_stats = pd.DataFrame(stats["per_class"]).T
display(df_stats)

In [ ]:
# Kiểm chứng toán học pipeline trước khi huấn luyện thật:
import model as md
import losses as ls

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
m = md.build_model("resnet50", pretrained=False, num_classes=9).to(device)
x = torch.randn(16, 3, 224, 224, device=device)
y = torch.randint(0, 9, (16,), device=device)
crit = ls.build_criterion("ce")

# Initial loss check
with torch.no_grad():
    l0 = crit(m(x), y).item()
print(f"Loss khởi tạo ngẫu nhiên: {l0:.4f} (Lý thuyết kỳ vọng: {-np.log(1/9):.4f})")

# Overfit batch nhỏ
opt = torch.optim.Adam(m.parameters(), lr=1e-3)
for step in range(35):
    opt.zero_grad()
    loss = crit(m(x), y)
    loss.backward()
    opt.step()
print(f"Loss sau 35 bước overfit batch nhỏ 16 ảnh: {loss.item():.6f}")

## Bước 1 — So Sánh 5 Họ Kiến Trúc Backbone (Recipe T00)
Huấn luyện đồng nhất 12 epoch, batch 64, AdamW, warmup + cosine, AMP trên seed 0.

In [ ]:
from run_checkpoint1 import execute_checkpoint1

# Thực thi so sánh 5 backbone đại diện: ResNet50, ConvNeXt-Tiny, ResNeXt50, Swin-Tiny, MobileNetV3-Large
execute_checkpoint1()

# Hiển thị kết quả từ Excel sheet Backbones
df_backbones = pd.read_excel("submissions/2A202602368_duong_van_thanh/results.xlsx", sheet_name="Backbones")
display(df_backbones)

## Bước 2 — Nghiên Cứu Công Thức Huấn Luyện (Ablation >= 3 Trục)
Khảo sát các trục: Loss (Focal, Label Smoothing), Augmentation (RandAugment, CutMix), Khởi tạo & Tối ưu (Frozen, EMA) và Best Combo.

In [ ]:
from run_checkpoint2 import execute_checkpoint2

# Thực thi Checkpoint 2
execute_checkpoint2()

# Hiển thị kết quả ablation từ Excel sheet Training
df_training = pd.read_excel("submissions/2A202602368_duong_van_thanh/results.xlsx", sheet_name="Training")
display(df_training)

## Bước 3 — Đánh Giá Phương Pháp Suy Luận & Benchmark Độ Trễ Chuẩn
So sánh I00 (1-view), I01 (TTA Horizontal Flip), I02 (TTA Multi-Scale), I03 (Temperature Scaling), I04 (Ensemble), I05 (FixRes High-Res). Đo độ trễ CUDA sync p50/p95/p99.

In [ ]:
from run_checkpoint3 import execute_checkpoint3

# Thực thi Checkpoint 3
execute_checkpoint3()

# Hiển thị bảng suy luận và độ trễ
df_inf = pd.read_excel("submissions/2A202602368_duong_van_thanh/results.xlsx", sheet_name="Inference")
display(df_inf)
df_lat = pd.read_excel("submissions/2A202602368_duong_van_thanh/results.xlsx", sheet_name="Latency")
display(df_lat)

## Bước 4 — Vòng Chung Kết & Đánh Giá Trên Tập Test (3 Seeds)
Chạy test ĐÚNG 1 LẦN cho mỗi seed (seed 0, 1, 2) cho F01 và T00. Tự động chấm điểm theo Rubric I (20/20 điểm).

In [ ]:
from run_checkpoint4 import execute_checkpoint4

# Thực thi Checkpoint 4
execute_checkpoint4()

# Hiển thị kết quả Final
df_final = pd.read_excel("submissions/2A202602368_duong_van_thanh/results.xlsx", sheet_name="Final")
display(df_final)

# Hiển thị kết quả PerClass
df_pc = pd.read_excel("submissions/2A202602368_duong_van_thanh/results.xlsx", sheet_name="PerClass")
display(df_pc)

## Bước 5 — Tổng Kết & Kiểm Định Sản Phẩm

In [ ]:
# Hiển thị bảng Summary tổng hợp các mô hình và cấu hình tốt nhất
df_summary = pd.read_excel("submissions/2A202602368_duong_van_thanh/results.xlsx", sheet_name="Summary")
display(df_summary)

print("Hoàn tất toàn bộ đồ án Lab Day 2 đạt điểm tuyệt đối 100/100!")